# تفسير القرطبي — Colab Notebook كامل
من تجهيز البيانات لحد تشغيل السيرفر وربطه بتطبيق المصحف عن طريق ngrok.

**الترتيب:**
1. تثبيت المكتبات
2. حط المفاتيح (Qdrant, Gemini, ngrok)
3. رفع ملف تفسير القرطبي (JSON) وتجهيزه على Qdrant — **مرة واحدة بس**
4. تشغيل سيرفر FastAPI + ngrok — كل مرة تفتح فيها الـ notebook


## 1) تثبيت المكتبات

In [1]:
!pip install -q fastapi uvicorn pydantic sentence-transformers qdrant-client google-genai pyngrok nest_asyncio tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 11.0 MB/s eta 0:00:0000:01


In [2]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.4 MB/s eta 0:00:00


## 2) المفاتيح
حط المفاتيح هنا مباشرة، أو الأفضل حطهم في **Colab Secrets** (أيقونة المفتاح 🔑 في الشريط الجانبي)
وبدّل السطور دي بـ `userdata.get("QDRANT_URL")` وهكذا.


In [4]:
import os
from google.colab import userdata

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
NGROK_TOKEN = userdata.get("NGROK_TOKEN")

os.environ["USE_CUDA"] = "1"
COLLECTION_NAME = "tafsir_qurtubi"
EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
EMBED_DIM = 1024

## 3) تجهيز البيانات ورفعها على Qdrant (مرة واحدة بس)

ارفع ملف تفسير القرطبي JSON (مثلاً من https://qul.tarteel.ai/resources/tafsir)
عن طريق أيقونة الملفات 📁 في الشريط الجانبي، أو شغّل الخلية اللي بعدها لرفعه مباشرة.

⚠️ لو already رفعت البيانات قبل كده على نفس الـ collection، **متشغلش** الخلايا دي تاني —
انزل على قسم "تشغيل السيرفر" على طول.


In [ ]:
from google.colab import files
uploaded = files.upload()  # اختار ملف الـ JSON من جهازك
DATA_PATH = list(uploaded.keys())[0]
print("اترفع:", DATA_PATH)

Saving ar-tafseer-al-qurtubi.json to ar-tafseer-al-qurtubi.json
اترفع: ar-tafseer-al-qurtubi.json


In [ ]:
import json
import re

def _strip_html(text):
    return re.sub(r"<[^>]+>", " ", text)

def load_records(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []
    if isinstance(data, dict):
        # شكل QUL / Tarteel AI: مفاتيح "surah:ayah"
        for key, val in data.items():
            if not isinstance(val, dict):
                continue
            text = _strip_html(val.get("text", "")).strip()
            if not text:
                continue
            ayah_keys = val.get("ayah_keys") or [key]
            for ak in ayah_keys:
                try:
                    surah_str, ayah_str = str(ak).split(":")
                    records.append({"surah": int(surah_str), "ayah": int(ayah_str), "text": text})
                except ValueError:
                    continue
    elif isinstance(data, list) and data and isinstance(data[0], dict):
        for item in data:
            text = item.get("text") or item.get("tafsir") or item.get("content")
            surah = item.get("surah") or item.get("surah_number") or item.get("sura")
            ayah = item.get("ayah") or item.get("ayah_number") or item.get("aya")
            if text and str(text).strip() and surah and ayah:
                records.append({"surah": int(surah), "ayah": int(ayah), "text": text})
    elif isinstance(data, list):
        for surah_idx, surah_ayat in enumerate(data, start=1):
            for ayah_idx, ayah_text in enumerate(surah_ayat, start=1):
                if ayah_text and str(ayah_text).strip():
                    records.append({"surah": surah_idx, "ayah": ayah_idx, "text": ayah_text})

    if not records:
        raise ValueError("ملف البيانات فاضي أو الشكل مش متعرّف عليه — راجع الشكل وعدّل الدالة")
    return records


def chunk_text(text: str, max_words: int = 200, overlap: int = 30) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks, start = [], 0
    while start < len(words):
        end = start + max_words
        chunks.append(" ".join(words[start:end]))
        start = end - overlap
    return chunks


records = load_records(DATA_PATH)
chunks = []
for r in records:
    for c_idx, chunk in enumerate(chunk_text(r["text"])):
        chunks.append({"surah": r["surah"], "ayah": r["ayah"], "chunk_index": c_idx, "text": chunk})

print(f"عدد الآيات: {len(records)} | عدد الـ chunks: {len(chunks)}")

عدد الآيات: 6234 | عدد الـ chunks: 32237


In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if os.environ.get("USE_CUDA") == "1" else "cpu")

texts_to_embed = ["passage: " + c["text"] for c in chunks]
all_embeddings = []
batch_size = 64
for i in tqdm(range(0, len(texts_to_embed), batch_size)):
    batch = texts_to_embed[i:i + batch_size]
    emb = embed_model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.extend(emb)

print("خلص الـ embedding:", len(all_embeddings))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

100%|██████████| 504/504 [50:00<00:00,  5.95s/it]

خلص الـ embedding: 32237


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

qdrant_client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=emb.tolist(),
        payload={"surah": c["surah"], "ayah": c["ayah"], "chunk_index": c["chunk_index"], "text": c["text"]},
    )
    for i, (c, emb) in enumerate(zip(chunks, all_embeddings))
]

for i in tqdm(range(0, len(points), 256)):
    qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points[i:i + 256])

print(f"✅ اترفع {len(points)} نقطة على collection '{COLLECTION_NAME}'")

/tmp/ipykernel_1273/1104557559.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(
100%|██████████| 126/126 [01:46<00:00,  1.19it/s]

✅ اترفع 32237 نقطة على collection 'tafsir_qurtubi'


## 4) تشغيل سيرفر FastAPI + ngrok

الخلايا دي شغّلها **كل مرة تفتح الـ notebook** عايز فيها التطبيق يوصل للسيرفر.
لو مفيش داعي تعيد رفع البيانات (خطوة 3)، تقدر تبدأ من هنا على طول بشرط تكون
شغّلت خلية المفاتيح فوق الأول.


In [ ]:
import os
import re
import traceback
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from groq import Groq

app = FastAPI(
    title="Quran AI Qurtubi Tafsir API",
    description="سيرفر تفسير آيات القرآن الكريم استناداً لتفسير القرطبي و Qdrant و Groq",
    version="1.2.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

if "embed_model" not in dir():
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if os.environ.get("USE_CUDA") == "1" else "cpu")
if "qdrant_client" not in dir():
    qdrant_client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

SCORE_THRESHOLD = 0.5
MAX_CHARS_PER_HIT = 800
MAX_CONTEXT_HITS = 3


# ======================= تنظيف وتنسيق النص =======================
_DIAC = re.compile(r"[\u064B-\u065F\u0670\u0640]")
_ROW_NUM = re.compile(r"\**\s*(\d+)\s*\**")


def clean_surah_name(name: str) -> str:
    parts = (name or "").strip().split(None, 1)
    if len(parts) == 2 and _DIAC.sub("", parts[0]) == "سورة":
        return parts[1]
    return (name or "").strip()


def _table_to_list(lines):
    rows = []
    for ln in lines:
        cells = [c.strip() for c in ln.strip().strip("|").split("|")]
        if all(re.fullmatch(r":?-{2,}:?", c) for c in cells if c):
            continue
        rows.append(cells)
    out = []
    for cells in rows[1:]:
        cells = [re.sub(r"\s*<br\s*/?>\s*", ": ", c, count=1) for c in cells]
        cells = [re.sub(r"\s*<br\s*/?>\s*", " ", c) for c in cells]
        cells = [c for c in cells if c]
        if not cells:
            continue
        m = _ROW_NUM.fullmatch(cells[0])
        if m and len(cells) > 1:
            out.append(f"{m.group(1)}. " + " — ".join(cells[1:]))
        else:
            out.append("- " + " — ".join(cells))
    return out


def clean_markdown(text: str) -> str:
    text = (text or "").replace("\r\n", "\n")

    # 0) شيل أي <think>...</think> لو الموديل رجّعها
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S | re.I)

    # 1) الجداول -> قوائم
    out, buf = [], []
    for ln in text.split("\n") + [""]:
        s = ln.strip()
        if s.startswith("|") and s.count("|") >= 2:
            buf.append(ln)
            continue
        if buf:
            out.extend(_table_to_list(buf))
            buf = []
        out.append(ln)
    text = "\n".join(out)

    # 2) وسوم HTML
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</?[a-zA-Z][^>]*>", "", text)

    # 3) الخطوط الفاصلة
    text = re.sub(r"^[ \t]*([-*_])\1{2,}[ \t]*$", "", text, flags=re.M)

    # 4) مسافات وأسطر فاضية زيادة
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 5) سطر فاضي قبل وبعد كل عنوان ###
    text = re.sub(r"([^\n])\n(#{1,6} )", r"\1\n\n\2", text)
    text = re.sub(r"^(#{1,6} .+)\n(?!\n)", r"\1\n\n", text, flags=re.M)
    return text.strip()


def markdown_to_plain(md: str) -> str:
    t = md or ""
    t = re.sub(r"^[ \t]{0,3}#{1,6}[ \t]*(.+?)[ \t]*#*[ \t]*$", r"\1", t, flags=re.M)
    t = re.sub(r"\*\*(.+?)\*\*", r"\1", t)
    t = re.sub(r"^[ \t]*[-*+][ \t]+", "• ", t, flags=re.M)
    t = re.sub(r"(?<![\w*])\*(?!\s)(.+?)(?<!\s)\*(?![\w*])", r"\1", t)
    t = re.sub(r"`([^`\n]*)`", r"\1", t)
    t = re.sub(r"^[ \t]*>[ \t]?", "", t, flags=re.M)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


# ======================= البرومبت =======================
FORMAT_RULES = """قواعد التنسيق (التزم بها حرفياً):
- لا تكرر نص الآية في إجابتك؛ هو معروض للمستخدم بالفعل.
- ابدأ مباشرة بالمحتوى بدون مقدمة أو ترحيب أو عنوان عام للسورة.
- ممنوع الجداول، وممنوع وسوم HTML مثل <br>، وممنوع الخطوط الفاصلة (---).
- العناوين الفرعية بصيغة ### فقط، وتحت كل عنوان فقرة قصيرة (من سطرين إلى 4 أسطر).
- القوائم الرقمية: كل بند في سطر واحد بالشكل: 1. **العنوان:** شرح في جملة أو جملتين.
- الإحالة لتفسير القرطبي بجملة قصيرة مثل «قال القرطبي: ...»، ولا تذكر أرقام سور أو آيات أخرى إلا إذا كانت متعلقة مباشرة بالآية.
- الطول الكلي حوالي 250 إلى 350 كلمة، بعربية فصيحة واضحة، بدون إيموجي أو زخارف."""

STRUCTURES = {
    "general": (
        "الهيكل المطلوب بالترتيب:\n"
        "### المعنى الإجمالي\nفقرة قصيرة تشرح المعنى.\n"
        "### {SRC}\nقائمة نقطية من 3 إلى 5 نقاط، كل نقطة في سطر واحد.\n"
        "### الخلاصة\nجملة أو جملتان."
    ),
    "linguistic": (
        "الهيكل المطلوب بالترتيب:\n"
        "### الإعراب\nنقاط قصيرة.\n"
        "### البلاغة والبيان\nنقاط قصيرة.\n"
        "### اللطائف اللغوية\nنقاط قصيرة."
    ),
    "lessons": (
        "الهيكل المطلوب بالترتيب:\n"
        "### الهدايات والدروس\nقائمة رقمية من 4 إلى 6 بنود بالشكل: 1. **العنوان:** الشرح.\n"
        "### الخلاصة\nجملة أو جملتان."
    ),
    "custom_question": (
        "الهيكل المطلوب:\n"
        "فقرتان أو ثلاث قصيرة تجيب مباشرة عن السؤال (بدون عنوان في البداية).\n"
        "### الخلاصة\nجملة واحدة."
    ),
}


def build_user_query(mode, surah_name, ayah_num, ayah_text, question):
    name = clean_surah_name(surah_name)
    if mode == "custom_question" and question:
        return f"السؤال حول الآية ({ayah_text}): {question}"
    if mode == "linguistic":
        return f"اشرح اللطائف اللغوية والإعراب والبلاغة لقوله تعالى: ({ayah_text}) في سورة {name} [آية {ayah_num}]."
    if mode == "lessons":
        return f"استخرج الهدايات والفوائد والدروس العملية المستفادة من قوله تعالى: ({ayah_text}) [سورة {name} - آية {ayah_num}]."
    return f"فسر لي الآية الكريمة: ({ayah_text}) من سورة {name} [آية رقم {ayah_num}] تفسيراً وافياً ميسراً."


def build_messages(user_query, context, mode, question):
    eff_mode = mode if (mode != "custom_question" or question) else "general"
    structure = STRUCTURES.get(eff_mode, STRUCTURES["general"])
    structure = structure.replace("{SRC}", "ما ذكره القرطبي" if context else "أبرز المعاني")
    system = FORMAT_RULES + "\n\n" + structure

    if context:
        user = (
            "أنت مساعد متخصص في تفسير القرآن الكريم، تعتمد فقط على تفسير القرطبي.\n\n"
            f"مقتطفات من تفسير القرطبي مرتبطة بالآية:\n{context}\n\n"
            f"الطلب: {user_query}\n\n"
            "أجب بالاعتماد على نصوص القرطبي أعلاه فقط."
        )
    else:
        user = (
            "لم يرد نص خاص بهذا الطلب في تفسير القرطبي المتاح لديك.\n"
            "أجب من معرفتك العامة، وابدأ بسطر واحد يوضح أن هذه الإجابة ليست من تفسير القرطبي مباشرة.\n\n"
            f"الطلب: {user_query}"
        )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


# ======================= البحث في Qdrant =======================
def retrieve_tafsir_for_ayah(surah: int, ayah: int, question: str = "", top_k: int = 5):
    query_text = question if question else f"سورة {surah} آية {ayah}"
    query_vector = embed_model.encode("query: " + query_text, normalize_embeddings=True)

    filter_condition = Filter(
        must=[
            FieldCondition(key="surah", match=MatchValue(value=surah)),
            FieldCondition(key="ayah", match=MatchValue(value=ayah)),
        ]
    )
    try:
        results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME, query=query_vector, query_filter=filter_condition, limit=top_k,
        ).points
        if not results:
            raise ValueError("no filtered hits")
    except Exception as e:
        print(f"⚠️ الفلتر على السورة {surah} / الآية {ayah} ما نفعش ({str(e)[:120]}) — بحث عام بدون فلتر")
        results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME, query=query_vector, limit=top_k,
        ).points
    return results


# راجع الأسماء دي من صفحة Models في Groq أو من: [m.id for m in groq_client.models.list().data]
LLM_MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.3-70b-versatile"]


def call_llm(messages):
    last_err = None
    for model in LLM_MODELS:
        try:
            r = groq_client.chat.completions.create(model=model, messages=messages)
            return r.choices[0].message.content or ""
        except Exception as e:  # 429 (حصة خلصت) أو موديل مش متاح
            print(f"⚠️ {model} فشل: {str(e)[:150]}")
            last_err = e
    raise last_err


def generate_tafsir_response(surah_num, ayah_num, surah_name, ayah_text, question=None, mode="general"):
    hits = retrieve_tafsir_for_ayah(surah=surah_num, ayah=ayah_num, question=question or ayah_text, top_k=5)

    # نتأكد إن المقطع فعلاً للآية المطلوبة (مهم لو الفلتر فشل وحصل بحث عام)
    relevant_hits = [
        h for h in hits
        if h.score >= SCORE_THRESHOLD
        and h.payload.get("surah") == surah_num
        and h.payload.get("ayah") == ayah_num
    ][:MAX_CONTEXT_HITS]

    context_parts = []
    for hit in relevant_hits:
        p = hit.payload
        text = p["text"][:MAX_CHARS_PER_HIT]
        context_parts.append(f"[سورة رقم {p['surah']} - آية رقم {p['ayah']}]\n{text}")
    context = "\n\n---\n\n".join(context_parts) if context_parts else None

    user_query = build_user_query(mode, surah_name, ayah_num, ayah_text, question)
    source = "قرطبي" if context else "عام (Groq)"

    raw = call_llm(build_messages(user_query, context, mode, question))
    return clean_markdown(raw), source


# ======================= الـ API =======================
class TafsirRequest(BaseModel):
    surah_number: int
    ayah_number: int
    surah_name: str
    ayah_text: str
    page_number: Optional[int] = None
    mode: Optional[str] = "general"
    question: Optional[str] = None
    model: Optional[str] = "qurtubi-qdrant-groq"


class TafsirResponse(BaseModel):
    tafsir: str
    tafsir_plain: str = ""
    surah_name: str
    surah_number: int
    ayah_number: int
    model: str = "qurtubi-qdrant-groq"
    source: str = "تفسير القرطبي (Qdrant + Groq RAG)"


@app.get("/health")
def health_check():
    return {"status": "ok", "message": "FastAPI Quran Tafsir RAG Server is running", "collection": COLLECTION_NAME}


@app.post("/api/tafsir", response_model=TafsirResponse)
def get_tafsir(req: TafsirRequest):  # def عادية: FastAPI بيشغّلها في thread pool فمش بتوقف السيرفر
    try:
        result_text, source = generate_tafsir_response(
            surah_num=req.surah_number, ayah_num=req.ayah_number,
            surah_name=req.surah_name, ayah_text=req.ayah_text,
            question=req.question, mode=req.mode or "general",
        )
        return TafsirResponse(
            tafsir=result_text,
            tafsir_plain=markdown_to_plain(result_text),
            surah_name=req.surah_name,
            surah_number=req.surah_number,
            ayah_number=req.ayah_number,
            source=source,
        )
    except Exception as e:
        print(traceback.format_exc())
        raise HTTPException(status_code=500, detail=str(e))

print("✅ الـ FastAPI app جاهزة")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ الـ FastAPI app جاهزة


In [ ]:
from pyngrok import ngrok
import uvicorn

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000).public_url
print("🟢 انسخ هذا الرابط وضعه في إعدادات تطبيق المصحف (تغيير رابط السيرفر):", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [1085]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🟢 انسخ هذا الرابط وضعه في إعدادات تطبيق المصحف (تغيير رابط السيرفر): https://keg-moody-running.ngrok-free.dev
INFO:     154.177.238.162:0 - "OPTIONS /api/tafsir HTTP/1.1" 200 OK
INFO:     154.177.238.162:0 - "POST /api/tafsir HTTP/1.1" 200 OK
INFO:     154.177.238.162:0 - "POST /api/tafsir HTTP/1.1" 200 OK
INFO:     154.177.238.162:0 - "POST /api/tafsir HTTP/1.1" 200 OK
INFO:     154.177.238.162:0 - "POST /api/tafsir HTTP/1.1" 200 OK
INFO:     154.177.238.162:0 - "POST /api/tafsir HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1085]


### ملاحظات
- الخلية اللي فوق **بتفضل شغالة** (السيرفر عامل block للتنفيذ) — سيبها شغالة طول ما محتاج التطبيق يوصل للسيرفر.
- عشان توقفها: `Runtime > Interrupt execution`.
- رابط ngrok المجاني بيتغيّر كل مرة تشغّل الخلية دي من جديد، فلو قطعت
  الاتصال هتحتاج تحدّث الرابط في التطبيق تاني.
- لو عايز رابط ثابت دايم بدل ما يتغيّر، أو عايز التطبيق يشتغل حتى لو
  الـ Colab مقفول، الحل الصح إنك تنشر نفس الكود ده على استضافة زي
  Render أو Railway بدل Colab.
